<a href="https://colab.research.google.com/github/HuVddme/speech_analysis/blob/main/Dementia_Regression_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/OpenBMB/VoxCPM.git

%cd VoxCPM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install voxcpm

In [ ]:
from modelscope import snapshot_download
snapshot_download('iic/speech_zipenhancer_ans_multiloss_16k_base')
snapshot_download('iic/SenseVoiceSmall')

from huggingface_hub import snapshot_download as hf_snapshot
hf_snapshot("openbmb/VoxCPM-0.5B")


1.Cell Setup

In [ ]:
!pip install -q transformers datasets torchaudio accelerate
!pip install -q faster-whisper
!pip install -q soundfile

import pandas as pd
import numpy as np
from pathlib import Path
import re
import os

from sklearn.model_selection import train_test_split

from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoFeatureExtractor,
    AutoTokenizer,
    AutoModelForAudioClassification,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import torchaudio
import torch
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

# verify gpu
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU is available and ready for training!")
else:
    print("Warning: GPU not found. Training will be slow.")


2.CREATE COMBINED MANIFEST FILES

In [ ]:
ROOT_DATA_PATH = Path('/content/drive/MyDrive/Dementia Analysis/dementia analysis/')
OUTPUT_PATH = ROOT_DATA_PATH / 'Manifest_Files'

DATASET_CONFIGS = [
    {
        'name': 'ADReSS',
        'path': ROOT_DATA_PATH / 'ADReSS-2020/ADReSS-IS2020-train/train',
        'sub_groups': ['cc', 'cd'],
        'audio_location': 'Full_wave_enhanced_audio',
        'transcript_location': 'transcription',
        'transcript_ext': '.cha'
    },
    {
        'name': 'ADReSSo_Diag',
        'path': ROOT_DATA_PATH / 'ADReSSo/ADReSSo-diagnosis-train/diagnosis/train',
        'sub_groups': ['ad', 'cn'],
        'audio_location': 'audio',
        'metadata_csv_path': ROOT_DATA_PATH / 'ADReSSo/ADReSSo-diagnosis-train/diagnosis/train/adresso-train-mmse-scores.csv'
    },
    {
        'name': 'ADReSS-M',
        'path': ROOT_DATA_PATH / 'ADReSS-M/ADReSS-M train/train',
        'sub_groups': [None],
    },
    {
        'name': 'TAUKADIAL',
        'path': ROOT_DATA_PATH / 'TAUKADIAL/TAUKADIAL-24-train/train',
        'sub_groups': [None],
    }
]

def infer_language(dataset_name: str) -> str:
    if dataset_name == "ADReSS-M":
        return "Greek"
    if dataset_name == "TAUKADIAL":
        return "Mandarin"
    return "English"

all_data = []
print("Starting combined data processing...")
print("=" * 60)

for config in DATASET_CONFIGS:
    dataset_name = config['name']
    dataset_path = config['path']
    language = infer_language(dataset_name)

    print(f"\n--> Processing Dataset: {dataset_name} ({language})")

    # Build search roots for audio
    search_paths = []
    if config['sub_groups'] == [None]:
        search_paths.append({'path': dataset_path, 'group': ''})
    else:
        for group in config['sub_groups']:
            audio_dir = dataset_path / config.get('audio_location', '') / group
            search_paths.append({'path': audio_dir, 'group': group})

    for item in search_paths:
        audio_dir = item['path']
        group_name = item['group']

        if not audio_dir.exists():
            print(f"  - Warning: Search directory not found: {audio_dir}")
            continue

        wav_files = list(audio_dir.rglob('*.wav'))
        mp3_files = list(audio_dir.rglob('*.mp3'))
        audio_files = wav_files + mp3_files

        if group_name:
            print(f"  - Group {group_name.upper()}: found {len(audio_files)} audio files.")
        else:
            print(f"  - Found {len(audio_files)} audio files recursively.")

        for audio_path in audio_files:
            try:
                base_id = audio_path.stem.strip()
                speaker_id = f"{dataset_name}__{base_id}"

                # Default text
                full_text = ""
                text_source = "missing"

                #Transcript Logic
                if dataset_name == 'ADReSS':
                    transcript_dir = dataset_path / config['transcript_location'] / group_name
                    transcript_path = transcript_dir / f"{base_id}{config['transcript_ext']}"

                    if transcript_path.exists():
                        with open(transcript_path, 'r', errors='ignore') as f:
                            lines = [line.strip() for line in f if line.startswith('*PAR:')]
                            # clean chat markers and delimiters
                            cleaned = []
                            for line in lines:
                                line = re.sub(r'^\*PAR:\s*', '', line)
                                line = re.sub(r'\s*\.+?\', '', line)
                                cleaned.append(line.strip())
                            full_text = " ".join([c for c in cleaned if c]).strip()
                        if full_text:
                            text_source = "manual"

                all_data.append({
                    'audio_filepath': str(audio_path),
                    'text': str(full_text),
                    'speaker_id': speaker_id,
                    'dataset': dataset_name,
                    'language': language,
                    'base_id': base_id,
                    'text_source': text_source
                })

            except Exception as e:
                print(f"  - Error processing {audio_path.name}: {e}")

print("\n" + "=" * 60)
print(f"Total processing complete. Found {len(all_data)} rows across all datasets.")

if all_data:
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(all_data)

    # Split by speaker_id
    unique_speakers = df[['speaker_id']].drop_duplicates()
    train_speakers, val_speakers = train_test_split(unique_speakers, test_size=0.2, random_state=42)

    train_df = df[df['speaker_id'].isin(train_speakers['speaker_id'])].reset_index(drop=True)
    val_df = df[df['speaker_id'].isin(val_speakers['speaker_id'])].reset_index(drop=True)

    train_manifest_path = OUTPUT_PATH / 'combined_multilingual_train_manifest.csv'
    val_manifest_path = OUTPUT_PATH / 'combined_multilingual_validation_manifest.csv'

    train_df.to_csv(train_manifest_path, index=False)
    val_df.to_csv(val_manifest_path, index=False)

    print(f"\n Created combined manifest files in: {OUTPUT_PATH}")
    print(f"  - Training Manifest: {train_manifest_path.name} ({len(train_df)} samples)")
    print(f"  - Validation Manifest: {val_manifest_path.name} ({len(val_df)} samples)")

    print("\nQuick check (language counts):")
    print(pd.concat([train_df, val_df])['language'].value_counts())
    print("\nQuick check (text_source counts):")
    print(pd.concat([train_df, val_df])['text_source'].value_counts())
else:
    print("\n No data was processed. Double-check your dataset paths.")


3.AST process

In [ ]:
from datasets import load_dataset, concatenate_datasets
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from pathlib import Path
import pandas as pd
import numpy as np

manifest_folder_path = '/content/drive/MyDrive/Dementia Analysis/dementia analysis/Manifest_Files/'
train_file = manifest_folder_path + 'combined_multilingual_train_manifest.csv'
val_file   = manifest_folder_path + 'combined_multilingual_validation_manifest.csv'

raw = load_dataset('csv', data_files={'train': train_file, 'validation': val_file})
full_raw = concatenate_datasets([raw["train"], raw["validation"]])
print("Manifest rows:", len(full_raw))

#AST Feature Extractor
model_checkpoint_ast = "Simon-Kotchou/ssast-small-patch-audioset-16-16"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_checkpoint_ast)
print("AST feature extractor loaded.")


ROOT_DATA_PATH = Path('/content/drive/MyDrive/Dementia Analysis/dementia analysis/')
all_meta_dfs = []

# ADReSS
p = ROOT_DATA_PATH / 'ADReSS-2020/ADReSS-IS2020-train/train/'
cc = pd.read_csv(p/'cc_meta_data.txt', sep=';', header=0, names=['ID','age','gender','MMSE'], na_values=[' NA'])
cd = pd.read_csv(p/'cd_meta_data.txt', sep=';', header=0, names=['ID','age','gender','MMSE'], na_values=[' NA'])
all_meta_dfs += [cc[['ID','MMSE']], cd[['ID','MMSE']]]

# ADReSSo
p = ROOT_DATA_PATH / 'ADReSSo/ADReSSo-diagnosis-train/diagnosis/train/adresso-train-mmse-scores.csv'
adresso = pd.read_csv(p).rename(columns={'adressfname':'ID','mmse':'MMSE'})
all_meta_dfs.append(adresso[['ID','MMSE']])

# ADReSS-M
p = ROOT_DATA_PATH / 'ADReSS-M/ADReSS-M train/training-groundtruth.csv'
m = pd.read_csv(p)

m = m.rename(columns={"adressfname": "ID", "mmse": "MMSE"})
m["ID"] = m["ID"].astype(str).str.strip()

all_meta_dfs.append(m[["ID", "MMSE"]])
print("  - ADReSS-M loaded:", m.shape)

# TAUKADIAL
p = ROOT_DATA_PATH / 'TAUKADIAL/TAUKADIAL-24-train/train/groundtruth.csv'
t = pd.read_csv(p)
for c in ["Id","tkdname","ID"]:
    if c in t.columns:
        t = t.rename(columns={c:"ID"})
        break
for c in ["MMSE","mmse"]:
    if c in t.columns:
        t = t.rename(columns={c:"MMSE"})
        break
all_meta_dfs.append(t[['ID','MMSE']])

meta = pd.concat(all_meta_dfs, ignore_index=True).dropna(subset=["MMSE"])
meta["ID"] = meta["ID"].astype(str).str.strip()
mmse_lookup = meta.set_index("ID")["MMSE"].to_dict()
print("MMSE lookup size:", len(mmse_lookup))

def lookup_mmse(dataset, base_id):
    base_id = str(base_id).strip()
    if dataset in ["ADReSS", "ADReSSo_Diag"]:
        return mmse_lookup.get(base_id)
    if dataset == "ADReSS-M":
        return mmse_lookup.get(base_id)
    if dataset == "TAUKADIAL":
        return mmse_lookup.get(base_id + ".wav") or mmse_lookup.get(base_id)
    return None

def add_label(ex):
    score = lookup_mmse(ex.get("dataset",""), ex.get("base_id",""))
    ex["label"] = float(score) if score is not None else -1.0
    return ex

labeled = full_raw.map(add_label)
labeled = labeled.filter(lambda x: x["label"] != -1.0)

print("AST labeled rows:", labeled.num_rows)
print(pd.Series(labeled["language"]).value_counts())


4.AST audio training

In [ ]:
import os, gc
import numpy as np
import pandas as pd
import soundfile as sf
import torchaudio
import torch
import torch.nn as nn

from pathlib import Path
from sklearn.model_selection import GroupKFold
from transformers import TrainingArguments, Trainer, AutoModelForAudioClassification

#config
SEED = 42
N_SPLITS = 5

EPOCHS_HEAD = 1
EPOCHS_FULL = 4

BATCH_TRAIN = 2
BATCH_EVAL = 4
GRAD_ACCUM = 8
LR = 1e-5
WEIGHT_DECAY = 0.01

MAX_SECONDS = 30

torch.manual_seed(SEED)
np.random.seed(SEED)


AST_OUT = ROOT_DATA_PATH / f"ssast-k{N_SPLITS}-huber-v4"
AST_OUT.mkdir(parents=True, exist_ok=True)

# Guard flag
done_flag = AST_OUT / "_DONE.flag"
if done_flag.exists():
    raise RuntimeError(f"Already finished AST k={N_SPLITS} run. Delete {done_flag} if you truly want to rerun.")

print("ROOT_DATA_PATH =", ROOT_DATA_PATH)
print("AST_OUT        =", AST_OUT.resolve())

# z-score
labels = np.array(labeled["label"], dtype=float)
mu, sigma = float(labels.mean()), float(labels.std() + 1e-8)
print(f"Label mean={mu:.3f}, std={sigma:.3f}")

def add_norm_label(ex):
    ex["label_norm"] = float((ex["label"] - mu) / sigma)
    return ex

labeled_norm = labeled.map(add_norm_label)

# METRICS
def rmse_mmse(y_true_norm, y_pred_norm):
    y_true = y_true_norm * sigma + mu
    y_pred = y_pred_norm * sigma + mu
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def compute_metrics(eval_pred):
    preds, y = eval_pred
    preds = np.asarray(preds).reshape(-1)
    y = np.asarray(y).reshape(-1)
    return {"rmse": rmse_mmse(y, preds)}


# AUDIO COLLATOR
target_sr = feature_extractor.sampling_rate
max_samples = int(MAX_SECONDS * target_sr) if MAX_SECONDS is not None else None

def audio_collator(features):
    audio_arrays = []
    y = []
    for f in features:
        wav, sr = sf.read(f["audio_filepath"])
        if wav.ndim > 1:
            wav = wav.mean(axis=1)

        waveform = torch.from_numpy(wav).float().unsqueeze(0)  # (1, T)
        if sr != target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, target_sr)

        if max_samples is not None and waveform.shape[-1] > max_samples:
            waveform = waveform[..., :max_samples]

        audio_arrays.append(waveform.squeeze(0).numpy())
        y.append(float(f["label_norm"]))

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=target_sr,
        padding=True,
        truncation=(max_samples is not None),
        max_length=(max_samples if max_samples is not None else None),
        return_tensors="pt"
    )
    inputs["labels"] = torch.tensor(y, dtype=torch.float32)
    return inputs


# TRAINER
class AudioRegressionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(-1)

        # Huber loss (robust to outliers)
        loss = nn.SmoothL1Loss(beta=1.0)(logits, labels)

        return (loss, outputs) if return_outputs else loss


groups = np.array(labeled_norm["speaker_id"])
kfold = GroupKFold(n_splits=N_SPLITS)

fold_rmses = []
rows = []

print(f"Starting AST regression | k={N_SPLITS} | seed={SEED} | max_seconds={MAX_SECONDS}")

for fold, (tr_idx, va_idx) in enumerate(kfold.split(np.zeros(len(groups)), groups=groups), start=1):
    print("\n" + "="*55)
    print(f"FOLD {fold}/{N_SPLITS} | Train={len(tr_idx)} | Val={len(va_idx)}")
    print("="*55)

    train_fold = labeled_norm.select(tr_idx.tolist())
    val_fold   = labeled_norm.select(va_idx.tolist())

    model = AutoModelForAudioClassification.from_pretrained(
        model_checkpoint_ast,
        num_labels=1,
        ignore_mismatched_sizes=True
    )
    # Make intent explicit
    model.config.problem_type = "regression"
    model.config.num_labels = 1

    print(model.config.model_type)
    print(model.config.hidden_size)

    #train head only
    for name, p in model.named_parameters():
        if "classifier" not in name:
            p.requires_grad = False

    args_head = TrainingArguments(
        output_dir=str(AST_OUT / f"fold-{fold:02d}-head"),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        report_to="none",
        num_train_epochs=EPOCHS_HEAD,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        logging_strategy="epoch",
        bf16=torch.cuda.is_available(),
        fp16=False,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        seed=SEED,
        load_best_model_at_end=True,
        metric_for_best_model="rmse",
        greater_is_better=False,
    )

    trainer_head = AudioRegressionTrainer(
        model=model,
        args=args_head,
        train_dataset=train_fold,
        eval_dataset=val_fold,
        data_collator=audio_collator,
        compute_metrics=compute_metrics,
    )
    trainer_head.train()

    del trainer_head
    torch.cuda.empty_cache()
    gc.collect()

    #full finetuning
    for p in model.parameters():
        p.requires_grad = True

    args_full = TrainingArguments(
        output_dir=str(AST_OUT / f"fold-{fold:02d}-full"),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        report_to="none",
        num_train_epochs=EPOCHS_FULL,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        logging_strategy="epoch",
        bf16=torch.cuda.is_available(),
        fp16=False,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        seed=SEED,
        load_best_model_at_end=True,
        metric_for_best_model="rmse",
        greater_is_better=False,
    )

    trainer_full = AudioRegressionTrainer(
        model=model,
        args=args_full,
        train_dataset=train_fold,
        eval_dataset=val_fold,
        data_collator=audio_collator,
        compute_metrics=compute_metrics,
    )
    trainer_full.train()

    #eval
    metrics = trainer_full.evaluate()
    rmse_val = float(metrics["eval_rmse"])
    fold_rmses.append(rmse_val)
    print("Fold RMSE (MMSE units):", rmse_val)

    #predictions
    pred = trainer_full.predict(val_fold)
    preds_norm = np.asarray(pred.predictions).reshape(-1)
    y_norm = np.asarray(pred.label_ids).reshape(-1)

    preds_mmse = preds_norm * sigma + mu
    y_mmse = y_norm * sigma + mu
    preds_mmse = np.clip(preds_mmse, 0.0, 30.0)

    for sid, yt, yp in zip(val_fold["speaker_id"], y_mmse, preds_mmse):
        rows.append({"fold": fold, "speaker_id": sid, "label": float(yt), "pred_audio": float(yp)})

    # saving final model
    fold_save_dir = AST_OUT / f"fold-{fold:02d}-final-v4"
    fold_save_dir.mkdir(parents=True, exist_ok=True)

    trainer_full.model.save_pretrained(fold_save_dir)
    feature_extractor.save_pretrained(fold_save_dir)

    # Save normalization stats
    (fold_save_dir / "label_norm_stats.txt").write_text(f"mu={mu}\n" f"sigma={sigma}\n")

    # Proof file
    (fold_save_dir / "_SAVED_OK.txt").write_text("saved\n")

    print("Saved fold final model to:", fold_save_dir.resolve())

    del trainer_full, model
    torch.cuda.empty_cache()
    gc.collect()

#saving csv output
df_preds = pd.DataFrame(rows)
pred_csv = AST_OUT / f"ast_k{N_SPLITS}_predictions.csv"
df_preds.to_csv(pred_csv, index=False)

rmse_csv = AST_OUT / f"ast_k{N_SPLITS}_fold_rmse.csv"
pd.DataFrame({"fold": list(range(1, N_SPLITS + 1)), "rmse": fold_rmses}).to_csv(rmse_csv, index=False)

# Done flag
done_flag.write_text("done")

print("\n" + "="*60)
print("AST k-fold complete")
print("Fold RMSEs:", [round(x, 4) for x in fold_rmses])
print(f"Mean RMSE: {float(np.mean(fold_rmses)):.4f} | Std: {float(np.std(fold_rmses)):.4f}")
print("Saved:")
print(" -", pred_csv.resolve())
print(" -", rmse_csv.resolve())
print("Guard flag written:", done_flag.resolve())

#list folder contents
print("\nAST_OUT contents:")
for p in sorted(AST_OUT.glob("*")):
    print(" -", p.name)

Speech to text transcripts

In [ ]:
from faster_whisper import WhisperModel
from tqdm.auto import tqdm


ROOT_DATA_PATH = Path('/content/drive/MyDrive/Dementia Analysis/dementia analysis/')
MANIFEST_PATH = ROOT_DATA_PATH / 'Manifest_Files'
train_manifest_path = MANIFEST_PATH / 'combined_multilingual_train_manifest.csv'
val_manifest_path   = MANIFEST_PATH / 'combined_multilingual_validation_manifest.csv'

ASR_CACHE_DIR = ROOT_DATA_PATH / "asr_transcripts"
ASR_CACHE_DIR.mkdir(parents=True, exist_ok=True)


train_df = pd.read_csv(train_manifest_path)
val_df   = pd.read_csv(val_manifest_path)
full_df  = pd.concat([train_df.assign(split="train"), val_df.assign(split="validation")], ignore_index=True)

print("Loaded manifests.")
print("Rows:", len(full_df))
print(full_df["language"].value_counts())
print(full_df["text_source"].value_counts())

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
print("ASR device:", device, "| compute_type:", compute_type)


asr_model = WhisperModel("medium", device=device, compute_type=compute_type)

def cache_path(row):
    return ASR_CACHE_DIR / f"{row['dataset']}__{row['base_id']}.txt"

def transcribe_one(audio_path, lang_hint):
    segments, info = asr_model.transcribe(
        audio_path,
        language=lang_hint,
        vad_filter=True
    )
    text = " ".join([seg.text.strip() for seg in segments]).strip()
    return text, info.language

def fill_asr(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    mask = (df["text_source"] == "missing") & (df["language"].isin(["Greek", "Mandarin"]))
    targets = df[mask].copy()

    print(f"ASR targets: {len(targets)} rows")

    for idx, row in tqdm(
    targets.iterrows(),
    total=len(targets),
    desc="Transcribing audio files"
):
        out = cache_path(row)

        # Load from cache if exists
        if out.exists():
            df.at[idx, "text"] = out.read_text(errors="ignore")
            df.at[idx, "text_source"] = "asr_cached"
            continue

        lang_hint = "el" if row["language"] == "Greek" else "zh"
        try:
            text, detected = transcribe_one(row["audio_filepath"], lang_hint=lang_hint)
            out.write_text(text)
            df.at[idx, "text"] = text
            df.at[idx, "text_source"] = f"asr_{detected}"
        except Exception as e:
            # Keep missing but record failure
            df.at[idx, "text"] = ""
            df.at[idx, "text_source"] = "asr_failed"
            print(f"ASR failed for {row['speaker_id']} | {row['audio_filepath']} | err={e}")

    return df

# Run ASR fill
full_df = fill_asr(full_df)

# Split back + save
train_out = full_df[full_df["split"] == "train"].drop(columns=["split"]).reset_index(drop=True)
val_out   = full_df[full_df["split"] == "validation"].drop(columns=["split"]).reset_index(drop=True)

train_out.to_csv(train_manifest_path, index=False)
val_out.to_csv(val_manifest_path, index=False)

print("\n Updated manifests saved with ASR transcripts.")
print("Updated text_source counts:")
print(pd.concat([train_out, val_out])["text_source"].value_counts())
print("\nNon-empty text counts by language:")
tmp = pd.concat([train_out, val_out])
tmp["has_text"] = tmp["text"].astype(str).str.strip().apply(lambda x: len(x) > 0)
print(tmp.groupby("language")["has_text"].sum())


5.ModernBERT process

In [ ]:
import pandas as pd
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer
from pathlib import Path

model_checkpoint_text = "jhu-clsp/mmBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_text)

ROOT_DATA_PATH = Path('/content/drive/MyDrive/Dementia Analysis/dementia analysis/')
MANIFEST_PATH = ROOT_DATA_PATH / 'Manifest_Files'

train_file = MANIFEST_PATH / 'combined_multilingual_train_manifest.csv'
val_file   = MANIFEST_PATH / 'combined_multilingual_validation_manifest.csv'


print("Loading metadata from all 4 datasets...")
all_meta_dfs = []

# ADReSS 2020
try:
    p = ROOT_DATA_PATH / 'ADReSS-2020/ADReSS-IS2020-train/train/'
    all_meta_dfs.append(pd.read_csv(p / 'cc_meta_data.txt', sep=';', header=0,
                                    names=['ID','age','gender','MMSE'], na_values=[' NA']))
    all_meta_dfs.append(pd.read_csv(p / 'cd_meta_data.txt', sep=';', header=0,
                                    names=['ID','age','gender','MMSE'], na_values=[' NA']))
    print("  - ADReSS loaded")
except Exception as e:
    print("  - ADReSS error:", e)

# ADReSSo
try:
    p = ROOT_DATA_PATH / 'ADReSSo/ADReSSo-diagnosis-train/diagnosis/train/adresso-train-mmse-scores.csv'
    df_adresso = pd.read_csv(p).rename(columns={'adressfname':'ID', 'mmse':'MMSE'})
    all_meta_dfs.append(df_adresso)
    print("  - ADReSSo loaded")
except Exception as e:
    print("  - ADReSSo error:", e)

# ADReSS-M
try:
    p = ROOT_DATA_PATH / 'ADReSS-M/ADReSS-M train/training-groundtruth.csv'
    df_m = pd.read_csv(p)
    if 'Id' in df_m.columns: df_m = df_m.rename(columns={'Id':'ID'})
    if 'mmse' in df_m.columns: df_m = df_m.rename(columns={'mmse':'MMSE'})
    all_meta_dfs.append(df_m[['ID','MMSE']])
    print("  - ADReSS-M loaded")
except Exception as e:
    print("  - ADReSS-M error:", e)

# TAUKADIAL
try:
    p = ROOT_DATA_PATH / 'TAUKADIAL/TAUKADIAL-24-train/train/groundtruth.csv'
    df_t = pd.read_csv(p)
    if 'Id' in df_t.columns: df_t = df_t.rename(columns={'Id':'ID'})
    if 'tkdname' in df_t.columns: df_t = df_t.rename(columns={'tkdname':'ID'})
    if 'mmse' in df_t.columns: df_t = df_t.rename(columns={'mmse':'MMSE'})
    all_meta_dfs.append(df_t[['ID','MMSE']])
    print("  - TAUKADIAL loaded")
except Exception as e:
    print("  - TAUKADIAL error:", e)

full_meta_df = pd.concat(all_meta_dfs, ignore_index=True)
full_meta_df = full_meta_df.dropna(subset=['MMSE']).copy()
full_meta_df['ID'] = full_meta_df['ID'].astype(str).str.strip()
mmse_lookup = full_meta_df.set_index('ID')['MMSE'].to_dict()
print(f"Metadata lookup ready: {len(mmse_lookup)} MMSE entries")


raw = load_dataset('csv', data_files={'train': str(train_file), 'validation': str(val_file)})
full_raw = concatenate_datasets([raw['train'], raw['validation']])
print("Manifest rows:", len(full_raw))

#label using base_id and specific datasets
def get_mmse_for_row(dataset, base_id):
    base_id = str(base_id).strip()

    if dataset in ["ADReSS", "ADReSSo_Diag"]:
        return mmse_lookup.get(base_id)

    if dataset == "ADReSS-M":
        return mmse_lookup.get(base_id)

    if dataset == "TAUKADIAL":
        return mmse_lookup.get(base_id + ".wav") or mmse_lookup.get(base_id)

    return None

def prep_row(ex):
    score = get_mmse_for_row(ex.get("dataset",""), ex.get("base_id",""))
    ex["label"] = float(score) if score is not None else -1.0
    ex["text"] = str(ex.get("text","") if ex.get("text") is not None else "")
    return ex

fixed = full_raw.map(prep_row)

clean = fixed.filter(lambda x: x["label"] != -1.0 and len(x["text"].strip()) > 0)

print("After filtering:")
print("Rows:", clean.num_rows)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=512)

tokenized = clean.map(tokenize_fn, batched=True)

df_check = tokenized.to_pandas()
print("\nLanguage counts in text dataset:")
print(df_check["language"].value_counts())
print("\nText source counts in text dataset:")
print(df_check["text_source"].value_counts())


7.ModernBERT train

In [ ]:
import os, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from transformers import (
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    AutoTokenizer
)

SEED = 42
N_SPLITS = 5

MODEL_CKPT = "jhu-clsp/mmBERT-base"
TEXT_OUT_DIR = ROOT_DATA_PATH / "mmbert-k5-huber-v3"
TEXT_OUT_DIR.mkdir(parents=True, exist_ok=True)

done_flag = TEXT_OUT_DIR / "_DONE.flag"
if done_flag.exists():
    raise RuntimeError(f"Already finished k={N_SPLITS}. Delete {done_flag} to rerun.")

MAX_LEN = 256
BATCH_TRAIN = 4
BATCH_EVAL = 8
GRAD_ACCUM = 4
EPOCHS = 6
LR = 2e-5

torch.manual_seed(SEED)
np.random.seed(SEED)


tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def retok(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

# ensure labels column
if "label" in tokenized.column_names and "labels" not in tokenized.column_names:
    tokenized = tokenized.rename_column("label", "labels")

tokenized256 = tokenized.map(retok, batched=True)
tokenized256 = tokenized256.shuffle(seed=SEED)

groups = np.array(tokenized256["speaker_id"])

tokenized256.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print(f"Dataset ready | rows={len(tokenized256)} | k={N_SPLITS}")


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.reshape(-1)
    labels = labels.reshape(-1)
    mse = mean_squared_error(labels, preds)
    return {"rmse": float(np.sqrt(mse))}

class RegressionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(-1)
        loss = nn.SmoothL1Loss(beta=1.0)(logits, labels)
        return (loss, outputs) if return_outputs else loss


kfold = GroupKFold(n_splits=N_SPLITS)

fold_rmses = []
rows = []

for fold, (tr_idx, va_idx) in enumerate(
    kfold.split(np.zeros(len(groups)), groups=groups), start=1
):
    print(f"\nFOLD {fold}/{N_SPLITS}")

    train_fold = tokenized256.select(tr_idx.tolist())
    val_fold   = tokenized256.select(va_idx.tolist())

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CKPT,
        num_labels=1,
        ignore_mismatched_sizes=True
    )

    args = TrainingArguments(
        output_dir=str(TEXT_OUT_DIR / f"fold-{fold:02d}"),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="rmse",
        greater_is_better=False,
        report_to="none",

        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        weight_decay=0.01,
        logging_strategy="epoch",

        bf16=True,
        fp16=False,
        seed=SEED,
    )

    trainer = RegressionTrainer(
        model=model,
        args=args,
        train_dataset=train_fold,
        eval_dataset=val_fold,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()
    rmse = float(metrics["eval_rmse"])
    fold_rmses.append(rmse)

    preds = trainer.predict(val_fold)
    yhat = preds.predictions.reshape(-1)
    y = preds.label_ids.reshape(-1)
    sids = np.array(tokenized256["speaker_id"])[va_idx]

    for sid, yt, yp in zip(sids, y, yhat):
        rows.append({
            "fold": fold,
            "speaker_id": sid,
            "label": float(yt),
            "pred_text": float(yp),
        })

    fold_save_dir = TEXT_OUT_DIR / f"fold-{fold:02d}-final"
    fold_save_dir.mkdir(parents=True, exist_ok=True)

    trainer.model.save_pretrained(fold_save_dir)
    tokenizer.save_pretrained(fold_save_dir)

    del trainer, model
    torch.cuda.empty_cache()
    gc.collect()

# SAVE OUTPUTS
pd.DataFrame(rows).to_csv(
    TEXT_OUT_DIR / "mmbert_k5_predictions.csv", index=False
)

pd.DataFrame({
    "fold": list(range(1, N_SPLITS + 1)),
    "rmse": fold_rmses
}).to_csv(
    TEXT_OUT_DIR / "mmbert_k5_fold_rmse.csv", index=False
)

done_flag.write_text("done")

print("mmBERT k-fold complete")
print("Mean RMSE:", float(np.mean(fold_rmses)))


Whisper

In [ ]:
# Whisper Setup
import os, gc
import numpy as np
import pandas as pd
import soundfile as sf
import torchaudio
import torch
import torch.nn as nn

from pathlib import Path
from sklearn.model_selection import GroupKFold
from transformers import (
    WhisperModel,
    WhisperProcessor,
    TrainingArguments,
    Trainer
)

# Checkpoint
WHISPER_CHECKPOINT = "openai/whisper-large-v3"

WHISPER_OUT = ROOT_DATA_PATH / f"whisper-k{N_SPLITS}-huber-v1"
WHISPER_OUT.mkdir(parents=True, exist_ok=True)

processor = WhisperProcessor.from_pretrained(WHISPER_CHECKPOINT)
whisper_model = WhisperModel.from_pretrained(WHISPER_CHECKPOINT)

def rmse_mmse(y_true_norm, y_pred_norm):
    y_true = y_true_norm * sigma + mu
    y_pred = y_pred_norm * sigma + mu
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def compute_metrics(eval_pred):
    preds, y = eval_pred
    preds = np.asarray(preds).reshape(-1)
    y = np.asarray(y).reshape(-1)
    return {"rmse": rmse_mmse(y, preds)}

# Guard flag
done_flag = WHISPER_OUT / "_DONE.flag"
if done_flag.exists():
    raise RuntimeError(f"Whisper already finished. Delete {done_flag} to rerun.")

print("WHISPER_OUT =", WHISPER_OUT.resolve())

for p in whisper_model.parameters():
    p.requires_grad = False

print("Whisper encoder frozen and moved to CUDA.")

if "labeled_norm" not in globals():
    labels = np.array(labeled["label"], dtype=float)
    mu, sigma = float(labels.mean()), float(labels.std() + 1e-8)

    def add_norm_label(ex):
        ex["label_norm"] = float((ex["label"] - mu) / sigma)
        return ex

    labeled_norm = labeled.map(add_norm_label)

    print("Recreated labeled_norm for Whisper")

print("Whisper encoder frozen.")
print("Whisper hidden size:", whisper_model.config.d_model)


In [ ]:
def whisper_audio_collator(features):
    audio_arrays = []
    labels = []

    for f in features:
        wav, sr = sf.read(f["audio_filepath"])
        if wav.ndim > 1:
            wav = wav.mean(axis=1)

        if sr != 16000:
            wav = torchaudio.functional.resample(
                torch.tensor(wav), sr, 16000
            ).numpy()

        audio_arrays.append(wav)
        labels.append(float(f["label_norm"]))

    inputs = processor(
        audio_arrays,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    x = inputs.input_features
    if x.shape[-1] < 3000:
        x = torch.nn.functional.pad(x, (0, 3000 - x.shape[-1]))
    x = x[:, :, :3000]

    inputs["input_features"] = x
    inputs["labels"] = torch.tensor(labels, dtype=torch.float32)
    return inputs

In [ ]:
class WhisperRegressor(nn.Module):
    def __init__(self, whisper):
        super().__init__()
        self.whisper = whisper

        dtype = next(whisper.parameters()).dtype
        device = next(whisper.parameters()).device

        self.reg_head = nn.Linear(
            whisper.config.d_model, 1
        ).to(device=device, dtype=dtype)

    def forward(self, input_features, labels=None):
        enc_out = self.whisper.encoder(
            input_features=input_features,
            return_dict=True
        )

        hidden = enc_out.last_hidden_state.mean(dim=1)  # [B, d_model]
        preds = self.reg_head(hidden).squeeze(-1)

        if labels is not None:
            loss = nn.SmoothL1Loss()(preds, labels)
            return {"loss": loss, "logits": preds}

        return {"logits": preds}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

groups = np.array(labeled_norm["speaker_id"])
kfold = GroupKFold(n_splits=N_SPLITS)

fold_rmses = []
rows = []

print(f"Starting Whisper regression | k={N_SPLITS} | seed={SEED}")

for fold, (tr_idx, va_idx) in enumerate(
    kfold.split(np.zeros(len(groups)), groups=groups), start=1
):
    print("\n" + "="*55)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("="*55)

    train_fold = labeled_norm.select(tr_idx.tolist())
    val_fold   = labeled_norm.select(va_idx.tolist())

    model = WhisperRegressor(whisper_model).to(device)

    args = TrainingArguments(
        output_dir=str(WHISPER_OUT / f"fold-{fold:02d}"),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        report_to="none",
        num_train_epochs=EPOCHS_FULL,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        logging_strategy="epoch",
        bf16=torch.cuda.is_available(),
        fp16=False,
        remove_unused_columns=False,
        seed=SEED,
        load_best_model_at_end=True,
        metric_for_best_model="rmse",
        greater_is_better=False,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_fold,
        eval_dataset=val_fold,
        data_collator=whisper_audio_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    metrics = trainer.evaluate()
    rmse_val = float(metrics["eval_rmse"])
    fold_rmses.append(rmse_val)
    print("Fold RMSE:", rmse_val)

    preds = trainer.predict(val_fold)
    preds_norm = preds.predictions.reshape(-1)
    y_norm = preds.label_ids.reshape(-1)

    preds_mmse = np.clip(preds_norm * sigma + mu, 0, 30)
    y_mmse = y_norm * sigma + mu

    for sid, yt, yp in zip(val_fold["speaker_id"], y_mmse, preds_mmse):
        rows.append({
            "fold": fold,
            "speaker_id": sid,
            "label": float(yt),
            "pred_whisper": float(yp),
        })

    del trainer, model
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
df_preds = pd.DataFrame(rows)
pred_csv = WHISPER_OUT / f"whisper_k{N_SPLITS}_predictions.csv"
df_preds.to_csv(pred_csv, index=False)

# Fold RMSE CSV
rmse_csv = WHISPER_OUT / f"whisper_k{N_SPLITS}_fold_rmse.csv"
pd.DataFrame({
    "fold": list(range(1, N_SPLITS + 1)),
    "rmse": fold_rmses
}).to_csv(rmse_csv, index=False)

# Summary stats
mean_rmse = float(np.mean(fold_rmses))
std_rmse  = float(np.std(fold_rmses))

done_flag = WHISPER_OUT / "_DONE.flag"
done_flag.write_text("done\n")

print("\n" + "="*60)
print("Whisper k-fold complete")
print("Fold RMSEs:", [round(x, 4) for x in fold_rmses])
print(f"Mean RMSE: {mean_rmse:.4f} | Std: {std_rmse:.4f}")
print("Saved:")
print(" -", pred_csv.resolve())
print(" -", rmse_csv.resolve())
print("Guard flag written:", done_flag.resolve())


RMSE by languages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

TEXT_PRED_PATH = ROOT_DATA_PATH / "mmbert-k5-huber-v3" / "mmbert_k5_predictions.csv"
AUDIO_PRED_PATH = ROOT_DATA_PATH / "ssast-k5-huber-v4" / "ast_k5_predictions.csv"
WHISPER_PRED_PATH = ROOT_DATA_PATH / "whisper-k5-huber-v1" / "whisper_k5_predictions.csv"

manifest_folder_path = Path(
    "/content/drive/MyDrive/Dementia Analysis/dementia analysis/Manifest_Files/"
)
train_file = manifest_folder_path / "combined_multilingual_train_manifest.csv"
val_file   = manifest_folder_path / "combined_multilingual_validation_manifest.csv"

def rmse(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def mae(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    return float(np.mean(np.abs(y - yhat)))

manifest_df = pd.concat(
    [pd.read_csv(train_file), pd.read_csv(val_file)],
    ignore_index=True
)
manifest_df = manifest_df[["speaker_id", "language"]].drop_duplicates()

df_text = pd.read_csv(TEXT_PRED_PATH)
df_audio = pd.read_csv(AUDIO_PRED_PATH)
df_whisper = pd.read_csv(WHISPER_PRED_PATH)

print("Loaded:")
print(" - Text   :", len(df_text))
print(" - Audio  :", len(df_audio))
print(" - Whisper:", len(df_whisper))

required_cols = {
    "text": ["speaker_id", "label", "pred_text"],
    "audio": ["speaker_id", "label", "pred_audio"],
    "whisper": ["speaker_id", "label", "pred_whisper"],
}

for name, cols in required_cols.items():
    df = locals()[f"df_{name}"]
    for c in cols:
        if c not in df.columns:
            raise ValueError(f"Missing '{c}' in {name} preds")

df = (
    df_text[["speaker_id", "label", "pred_text"]]
    .merge(
        df_audio[["speaker_id", "pred_audio"]],
        on="speaker_id",
        how="inner"
    )
    .merge(
        df_whisper[["speaker_id", "pred_whisper"]],
        on="speaker_id",
        how="inner"
    )
)

df["label"] = df["label"].astype(float)

df = df.merge(manifest_df, on="speaker_id", how="left")

print("\nCommon samples:", len(df))
print("Language distribution:")
print(df["language"].value_counts(dropna=False))


overall = pd.DataFrame([
    {
        "model": "SSAST (audio)",
        "RMSE": rmse(df["label"], df["pred_audio"]),
        "MAE": mae(df["label"], df["pred_audio"]),
    },
    {
        "model": "MM-BERT (text)",
        "RMSE": rmse(df["label"], df["pred_text"]),
        "MAE": mae(df["label"], df["pred_text"]),
    },
    {
        "model": "Whisper (speech)",
        "RMSE": rmse(df["label"], df["pred_whisper"]),
        "MAE": mae(df["label"], df["pred_whisper"]),
    },
]).sort_values("RMSE")

print("\n=== Overall Performance (Common Samples) ===")
print(overall.to_string(index=False))

rows = []
for lang, g in df.groupby("language"):
    rows.append({
        "language": lang,
        "n": len(g),
        "rmse_ssast": rmse(g["label"], g["pred_audio"]),
        "rmse_mmbert": rmse(g["label"], g["pred_text"]),
        "rmse_whisper": rmse(g["label"], g["pred_whisper"]),
    })

rmse_lang = pd.DataFrame(rows).sort_values("language")

print("\n=== RMSE by Language (Common Samples) ===")
print(rmse_lang.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

xmin, xmax = 0, 30
ymin, ymax = 0, 30

languages = sorted(df["language"].dropna().unique())

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

ax = axes[0]
for lang in languages:
    g = df[df["language"] == lang]
    ax.scatter(g["label"], g["pred_audio"], alpha=0.6, label=lang)

ax.plot([xmin, xmax], [ymin, ymax], "k--")
ax.set_title("SSAST (Audio)\nPredicted vs Actual")
ax.set_xlabel("Actual MMSE")
ax.set_ylabel("Predicted MMSE")
ax.grid(True)

ax = axes[1]
for lang in languages:
    g = df[df["language"] == lang]
    ax.scatter(g["label"], g["pred_text"], alpha=0.6, label=lang)

ax.plot([xmin, xmax], [ymin, ymax], "k--")
ax.set_title("MM-BERT (Text)\nPredicted vs Actual")
ax.set_xlabel("Actual MMSE")
ax.grid(True)

ax = axes[2]
for lang in languages:
    g = df[df["language"] == lang]
    ax.scatter(g["label"], g["pred_whisper"], alpha=0.6, label=lang)

ax.plot([xmin, xmax], [ymin, ymax], "k--")
ax.set_title("Whisper \nPredicted vs Actual")
ax.set_xlabel("Actual MMSE")
ax.grid(True)

ax = axes[3]
for lang in languages:
    g = df[df["language"] == lang]
    ax.scatter(g["pred_audio"], g["pred_text"], alpha=0.6, label=lang)

ax.set_xlabel("SSAST Prediction")
ax.set_ylabel("MM-BERT Prediction")
ax.set_title("Audio vs Text Predictions")
ax.grid(True)

ax = axes[4]
for lang in languages:
    g = df[df["language"] == lang]
    ax.scatter(g["pred_whisper"], g["pred_text"], alpha=0.6, label=lang)

ax.set_xlabel("Whisper Prediction")
ax.set_ylabel("MM-BERT Prediction")
ax.set_title("Whisper vs Text Predictions")
ax.grid(True)

ax = axes[5]
for lang in languages:
    g = df[df["language"] == lang]
    residuals = g["pred_text"] - g["label"]
    ax.scatter(g["label"], residuals, alpha=0.6, label=lang)

ax.axhline(0, color="k", linestyle="--")
ax.set_xlabel("Actual MMSE")
ax.set_ylabel("Residual (Text)")
ax.set_title("MM-BERT Residuals by Language")
ax.grid(True)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Language", loc="upper center", ncol=6)

plt.suptitle("Multimodal Scatter Diagnostics by Language", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


Concat Fusion

In [ ]:
import torch
from transformers import (
    AutoModelForAudioClassification,
    AutoFeatureExtractor,
    AutoTokenizer,
    AutoModelForSequenceClassification
)

#AST
feature_extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)
AST_CHECKPOINT = "/content/drive/MyDrive/Dementia Analysis/dementia analysis/ast-k5-huber-v3/fold-01-final-v3"

ast_model = AutoModelForAudioClassification.from_pretrained(
    AST_CHECKPOINT,
    local_files_only=True
)


#ModernBERT
tokenizer = AutoTokenizer.from_pretrained(
    "jhu-clsp/mmBERT-base"
)

TEXT_CHECKPOINT = "/content/drive/MyDrive/Dementia Analysis/dementia analysis/mmbert-k5-huber-v3/fold-01-final"
text_model = AutoModelForSequenceClassification.from_pretrained(
    TEXT_CHECKPOINT,
    local_files_only=True
)

print("✅ AST and ModernBERT models loaded")


In [ ]:
import pandas as pd
from pathlib import Path

FUSION_CSV = ROOT_DATA_PATH / "fusion_stepA_common_samples.csv"
df_base = pd.read_csv(FUSION_CSV)

print("Loaded base fusion:", df_base.shape)
print("Columns:", df_base.columns.tolist())

# Load manifests to attach text + audio paths
manifest_folder_path = Path('/content/drive/MyDrive/Dementia Analysis/dementia analysis/Manifest_Files/')
train_file = manifest_folder_path / 'combined_multilingual_train_manifest.csv'
val_file   = manifest_folder_path / 'combined_multilingual_validation_manifest.csv'

manifest_df = pd.concat([pd.read_csv(train_file), pd.read_csv(val_file)], ignore_index=True)

need = {"speaker_id", "audio_filepath", "text"}
missing = need - set(manifest_df.columns)
if missing:
    raise ValueError(f"Manifest missing columns {missing}. Found: {manifest_df.columns.tolist()}")

# Merge
df_fusion = df_base.merge(
    manifest_df[["speaker_id", "audio_filepath", "text", "language"]],
    on="speaker_id",
    how="left"
)

print("After merge:", df_fusion.shape)
print("Columns after merge:", df_fusion.columns.tolist())
print("Missing audio_filepath:", df_fusion["audio_filepath"].isna().sum())
print("Missing text:", df_fusion["text"].isna().sum())

if "language" not in df_fusion.columns:
    lang_cols = [c for c in df_fusion.columns if c.startswith("language")]
    print("Found language-like columns:", lang_cols)
    if "language_x" in df_fusion.columns:
        df_fusion["language"] = df_fusion["language_x"]
    elif "language_y" in df_fusion.columns:
        df_fusion["language"] = df_fusion["language_y"]
    else:
        df_fusion["language"] = df_fusion[lang_cols[0]]

if "label" not in df_fusion.columns:
    if "label_text" in df_fusion.columns:
        df_fusion["label"] = df_fusion["label_text"]
    else:
        raise ValueError("No label column found.")

df_fusion = df_fusion[["speaker_id", "label", "language", "audio_filepath", "text"]].copy()

print("Final df_fusion:", df_fusion.shape)
df_fusion.head()


In [ ]:
import soundfile as sf
import torchaudio
import torch
from torch.utils.data import Dataset, DataLoader

class FusionDataset(Dataset):
    def __init__(self, df, tokenizer, feature_extractor, max_len=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.feat = feature_extractor
        self.max_len = max_len
        self.target_sr = feature_extractor.sampling_rate

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # TEXT (NO padding here)
        text_inputs = self.tokenizer(
            str(row["text"]),
            truncation=True,
            max_length=self.max_len,
            return_tensors=None,
        )

        # AUDIO
        wav, sr = sf.read(row["audio_filepath"])
        if wav.ndim > 1:
            wav = wav.mean(axis=1)

        wav = torch.tensor(wav, dtype=torch.float32).unsqueeze(0)
        if sr != self.target_sr:
            wav = torchaudio.functional.resample(wav, sr, self.target_sr)

        # feature_extractor expects numpy 1D arrays
        audio_inputs = self.feat(
            wav.squeeze(0).numpy(),
            sampling_rate=self.target_sr,
            return_tensors="pt",
        )

        return {
            "text_inputs": text_inputs,
            "audio_inputs": {k: v.squeeze(0) for k, v in audio_inputs.items()},
            "label": float(row["label"]),
            "speaker_id": row["speaker_id"],
            "language": row["language"],
        }

def fusion_collate(batch):
    # TEXT pad dynamically
    texts = [b["text_inputs"] for b in batch]
    text_inputs = tokenizer.pad(
        texts,
        padding=True,
        return_tensors="pt"
    )

    # AUDIO stack
    audio_inputs = {
        k: torch.stack([b["audio_inputs"][k] for b in batch])
        for k in batch[0]["audio_inputs"]
    }

    labels = torch.tensor([b["label"] for b in batch], dtype=torch.float32)
    speaker_ids = [b["speaker_id"] for b in batch]
    languages = [b["language"] for b in batch]

    return {
        "text_inputs": text_inputs,
        "audio_inputs": audio_inputs,
        "label": labels,
        "speaker_id": speaker_ids,
        "language": languages
    }

fusion_dataset = FusionDataset(df_fusion, tokenizer, feature_extractor, max_len=256)

fusion_loader = DataLoader(
    fusion_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=fusion_collate,
    num_workers=0
)

print("Fusion DataLoader ready | batches:", len(fusion_loader))

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc
import torch

rows = []
ast_model.to(device).eval()
text_model.to(device).eval()

with torch.inference_mode():
    for batch in tqdm(fusion_loader, desc="Extracting embeddings (AST + Text)"):
        audio_inputs = {k: v.to(device) for k, v in batch["audio_inputs"].items()}

        ast_out = ast_model(**audio_inputs, output_hidden_states=True, return_dict=True)

        if hasattr(ast_out, "pooler_output") and ast_out.pooler_output is not None:
            audio_emb = ast_out.pooler_output
        elif hasattr(ast_out, "hidden_states") and len(ast_out.hidden_states) > 0:
            audio_emb = ast_out.hidden_states[-1][:, 0, :]
        else:
            raise AttributeError("Could not extract audio embeddings from AST model output.")

        text_inputs = {k: v.to(device) for k, v in batch["text_inputs"].items()}

        txt_out = text_model(**text_inputs, output_hidden_states=True, return_dict=True)

        if hasattr(txt_out, "pooler_output") and txt_out.pooler_output is not None:
            text_emb = txt_out.pooler_output
        elif hasattr(txt_out, "hidden_states") and len(txt_out.hidden_states) > 0:
            text_emb = txt_out.hidden_states[-1][:, 0, :]
        else:
            raise AttributeError("Could not extract text embeddings from ModernBERT model output.")

        audio_emb = audio_emb.float().cpu().numpy()
        text_emb  = text_emb.float().cpu().numpy()

        labels = batch["label"].cpu().numpy()
        speaker_ids = batch["speaker_id"]
        langs = batch["language"]

        for i in range(len(speaker_ids)):
            rows.append({
                "speaker_id": speaker_ids[i],
                "language": langs[i],
                "label": float(labels[i]),
                "audio_emb": audio_emb[i],
                "text_emb": text_emb[i],
            })

        del audio_inputs, text_inputs, ast_out, txt_out, audio_emb, text_emb
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df_emb = pd.DataFrame(rows)
print("Extracted embeddings:", df_emb.shape)
print("Emb sizes:", len(df_emb.iloc[0]["audio_emb"]), len(df_emb.iloc[0]["text_emb"]))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

X_audio = np.stack(df_emb["audio_emb"].values)
X_text  = np.stack(df_emb["text_emb"].values)
y       = df_emb["label"].values.astype(float)

# CONCAT FUSION FEATURES
X_concat = np.concatenate([X_audio, X_text], axis=1)
print("X_concat shape:", X_concat.shape)

# Save features
OUT_DIR = ROOT_DATA_PATH / "concat_fusion"
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUT_DIR / "X_concat.npy", X_concat)
np.save(OUT_DIR / "y.npy", y)
df_emb[["speaker_id", "language"]].to_csv(OUT_DIR / "meta.csv", index=False)

print("Saved:")
print(" -", OUT_DIR / "X_concat.npy")
print(" -", OUT_DIR / "y.npy")
print(" -", OUT_DIR / "meta.csv")


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

import matplotlib.pyplot as plt
from pathlib import Path
import gc

SEED = 42
N_SPLITS = 5

BATCH = 64 if torch.cuda.is_available() else 32
EPOCHS = 40
PATIENCE = 7

LR = 2e-4
WEIGHT_DECAY = 1e-2
HIDDEN = 512
DROPOUT = 0.25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

torch.manual_seed(SEED)
np.random.seed(SEED)

OUT_DIR = ROOT_DATA_PATH / "concat_fusion"
X = np.load(OUT_DIR / "X_concat.npy")
y = np.load(OUT_DIR / "y.npy").astype(np.float32)
meta = pd.read_csv(OUT_DIR / "meta.csv")

groups = meta["speaker_id"].astype(str).values
languages = meta["language"].values

print("X:", X.shape, "y:", y.shape)

class NpDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class ConcatMLP(nn.Module):
    def __init__(self, in_dim, hidden=512, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def train_one_fold(model, train_loader, val_loader):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    loss_fn = nn.SmoothL1Loss(beta=1.0)

    best_rmse = float("inf")
    best_state = None
    bad = 0

    for epoch in range(1, EPOCHS + 1):

        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        #validate
        model.eval()
        preds_all, ys_all = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                preds = model(xb).cpu().numpy()
                preds_all.append(preds)
                ys_all.append(yb.numpy())

        preds_all = np.concatenate(preds_all)
        ys_all = np.concatenate(ys_all)

        val_rmse = rmse(ys_all, preds_all)

        if val_rmse < best_rmse - 1e-4:
            best_rmse = val_rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_rmse

#training
gkf = GroupKFold(n_splits=N_SPLITS)

oof_pred = np.zeros_like(y, dtype=np.float32)
fold_rmses = []

for fold, (tr, va) in enumerate(gkf.split(X, y, groups=groups), start=1):
    print(f"\n=== Fold {fold}/{N_SPLITS} ===")

    train_ds = NpDataset(X[tr], y[tr])
    val_ds   = NpDataset(X[va], y[va])

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False)

    model = ConcatMLP(
        in_dim=X.shape[1],
        hidden=HIDDEN,
        dropout=DROPOUT
    ).to(DEVICE)

    best_rmse = train_one_fold(model, train_loader, val_loader)
    fold_rmses.append(best_rmse)

    # OOF predictions
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            xb = xb.to(DEVICE)
            preds.append(model(xb).cpu().numpy())
    preds = np.concatenate(preds)

    oof_pred[va] = preds
    print(f"Fold RMSE: {best_rmse:.4f}")

    del model
    torch.cuda.empty_cache()
    gc.collect()

#FINAL METRICS
oof_rmse = rmse(y, oof_pred)
oof_mae  = mae(y, oof_pred)

print("\n=== CONCAT FUSION (MLP + Huber) RESULTS ===")
print("Fold RMSE mean:", float(np.mean(fold_rmses)))
print("Fold RMSE std :", float(np.std(fold_rmses)))
print("OOF RMSE      :", oof_rmse)
print("OOF MAE       :", oof_mae)

# SAVE OUTPUTS
SAVE_DIR = ROOT_DATA_PATH / "concat-mlp-k5-huber"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

df_out = pd.DataFrame({
    "speaker_id": groups,
    "language": languages,
    "label": y,
    "pred_concat": oof_pred,
})

df_out.to_csv(SAVE_DIR / "concat_mlp_k5_predictions.csv", index=False)
pd.DataFrame({"fold": list(range(1, N_SPLITS + 1)), "rmse": fold_rmses}) \
  .to_csv(SAVE_DIR / "concat_mlp_k5_fold_rmse.csv", index=False)

print("Saved to:", SAVE_DIR)

# SCATTER PLOT
plt.figure(figsize=(7,7))
plt.scatter(y, np.clip(oof_pred, 0, 30), alpha=0.7)
plt.plot([0, 30], [0, 30], "r--", label="Perfect Prediction")
plt.xlim(0, 30)
plt.ylim(0, 30)
plt.grid(True)
plt.xlabel("Actual MMSE")
plt.ylabel("Predicted MMSE")
plt.title(f"Concat Fusion ={oof_rmse:.2f}")
plt.legend()
plt.show()


MLP + GATED fusion

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLPEncoder(nn.Module):
    def __init__(self, in_dim: int, hid_dim: int, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hid_dim, hid_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class GatedFusionRegressor(nn.Module):
    def __init__(
        self,
        text_in_dim: int,
        audio_in_dim: int,
        hid_dim: int = 256,
        dropout: float = 0.2,
        gate_hidden: int = 128,
    ):
        super().__init__()
        self.text_enc = MLPEncoder(text_in_dim, hid_dim, dropout)
        self.audio_enc = MLPEncoder(audio_in_dim, hid_dim, dropout)

        self.gate = nn.Sequential(
            nn.Linear(2 * hid_dim, gate_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(gate_hidden, hid_dim),
            nn.Sigmoid(),
        )

        self.head = nn.Sequential(
            nn.Linear(hid_dim, hid_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hid_dim // 2, 1),
        )

    def forward(self, x_text, x_audio):
        ht = self.text_enc(x_text)
        ha = self.audio_enc(x_audio)
        g = self.gate(torch.cat([ht, ha], dim=-1))
        h = g * ht + (1.0 - g) * ha
        yhat = self.head(h).squeeze(-1)
        return yhat, g


In [ ]:
def gate_regularizer(g, strength=1e-3):
    return strength * ((g - 0.5) ** 2).mean()

In [ ]:
from torch.utils.data import DataLoader

def train_one_epoch(model, loader, optimizer, device, loss_type="l1", gate_reg=0.0, mod_drop_p=0.0):
    model.train()
    total_loss = 0.0

    for batch in loader:
        x_text = batch["x_text"].to(device)
        x_audio = batch["x_audio"].to(device)
        y = batch["y"].to(device).float()

        optimizer.zero_grad(set_to_none=True)

        # optional modality dropout
        if mod_drop_p > 0.0:
            if torch.rand(1).item() < mod_drop_p:
                x_text = torch.zeros_like(x_text)
            if torch.rand(1).item() < mod_drop_p:
                x_audio = torch.zeros_like(x_audio)

        yhat, g = model(x_text, x_audio)

        if loss_type == "mse":
            loss = F.mse_loss(yhat, y)
        else:
            loss = F.l1_loss(yhat, y)

        if gate_reg > 0:
            loss = loss + gate_regularizer(g, strength=gate_reg)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_rmse(model, loader, device):
    model.eval()
    ys, preds = [], []
    for batch in loader:
        x_text = batch["x_text"].to(device)
        x_audio = batch["x_audio"].to(device)
        y = batch["y"].to(device).float()

        yhat, _ = model(x_text, x_audio)
        ys.append(y.cpu())
        preds.append(yhat.cpu())

    ys = torch.cat(ys).numpy()
    preds = torch.cat(preds).numpy()
    rmse = float(((ys - preds) ** 2).mean() ** 0.5)
    return rmse


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import gc
from pathlib import Path

SEED = 42
N_SPLITS = 5

BATCH = 64 if torch.cuda.is_available() else 32
EPOCHS = 40
PATIENCE = 7

LR = 2e-4
WEIGHT_DECAY = 1e-2
HIDDEN = 512
DROPOUT = 0.25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

# LOAD EMBEDDINGS
OUT_DIR = ROOT_DATA_PATH / "concat_fusion"
X_audio = np.load(OUT_DIR / "X_concat.npy")[:, :768]
X_text  = np.load(OUT_DIR / "X_concat.npy")[:, 768:]
y = np.load(OUT_DIR / "y.npy").astype(np.float32)
meta = pd.read_csv(OUT_DIR / "meta.csv")

groups = meta["speaker_id"].astype(str).values
languages = meta["language"].values

class FusionDataset(Dataset):
    def __init__(self, Xa, Xt, y):
        self.Xa = torch.from_numpy(Xa).float()
        self.Xt = torch.from_numpy(Xt).float()
        self.y  = torch.from_numpy(y).float()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.Xa[i], self.Xt[i], self.y[i]

# MODEL
class GatedFusionMLP(nn.Module):
    def __init__(self, dim=768, hidden=512, dropout=0.25):
        super().__init__()

        self.audio_proj = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden),
            nn.GELU()
        )
        self.text_proj = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden),
            nn.GELU()
        )

        self.gate = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.Sigmoid()
        )

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, xa, xt):
        ha = self.audio_proj(xa)
        ht = self.text_proj(xt)
        g  = self.gate(torch.cat([ha, ht], dim=1))
        h  = g * ha + (1 - g) * ht
        return self.head(h).squeeze(-1)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def train_one_fold(model, tr_loader, va_loader):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.SmoothL1Loss(beta=1.0)

    best_rmse, bad = float("inf"), 0
    best_state = None

    for _ in range(EPOCHS):
        model.train()
        for xa, xt, yb in tr_loader:
            xa, xt, yb = xa.to(DEVICE), xt.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xa, xt), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        preds, ys = [], []
        with torch.no_grad():
            for xa, xt, yb in va_loader:
                xa, xt = xa.to(DEVICE), xt.to(DEVICE)
                preds.append(model(xa, xt).cpu().numpy())
                ys.append(yb.numpy())

        preds = np.concatenate(preds)
        ys = np.concatenate(ys)
        val_rmse = rmse(ys, preds)

        if val_rmse < best_rmse - 1e-4:
            best_rmse = val_rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return best_rmse

gkf = GroupKFold(n_splits=N_SPLITS)
oof_pred = np.zeros_like(y)
fold_rmses = []

for fold, (tr, va) in enumerate(gkf.split(X_audio, y, groups=groups), 1):
    print(f"\n=== Fold {fold}/{N_SPLITS} ===")

    tr_ds = FusionDataset(X_audio[tr], X_text[tr], y[tr])
    va_ds = FusionDataset(X_audio[va], X_text[va], y[va])

    tr_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH, shuffle=False)

    model = GatedFusionMLP().to(DEVICE)
    best_rmse = train_one_fold(model, tr_loader, va_loader)
    fold_rmses.append(best_rmse)

    model.eval()
    with torch.no_grad():
        preds = []
        for xa, xt, _ in va_loader:
            preds.append(model(xa.to(DEVICE), xt.to(DEVICE)).cpu().numpy())
    oof_pred[va] = np.concatenate(preds)

    print(f"Fold RMSE: {best_rmse:.4f}")

    del model
    torch.cuda.empty_cache()
    gc.collect()


print("\n=== GATED FUSION RESULTS ===")
print("Mean RMSE:", np.mean(fold_rmses))
print("Std  RMSE:", np.std(fold_rmses))
print("OOF RMSE :", rmse(y, oof_pred))
print("OOF MAE  :", mae(y, oof_pred))


SAVE_DIR = ROOT_DATA_PATH / "gated-mlp-k5-huber"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "speaker_id": groups,
    "language": languages,
    "label": y,
    "pred_gated": oof_pred
}).to_csv(SAVE_DIR / "gated_k5_predictions.csv", index=False)

plt.figure(figsize=(7,7))
plt.scatter(y, np.clip(oof_pred, 0, 30), alpha=0.7)
plt.plot([0,30],[0,30],'r--')
plt.xlabel("Actual MMSE")
plt.ylabel("Predicted MMSE")
plt.title(f"Gated Fusion | RMSE={rmse(y,oof_pred):.2f}")
plt.grid(True)
plt.show()


Cross-Attention fusion

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import gc
from pathlib import Path

SEED = 42
N_SPLITS = 5

BATCH = 64 if torch.cuda.is_available() else 32
EPOCHS = 40
PATIENCE = 7

LR = 2e-4
WEIGHT_DECAY = 1e-2
HIDDEN = 256
DROPOUT = 0.25
N_HEADS = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)


OUT_DIR = ROOT_DATA_PATH / "concat_fusion"
X_concat = np.load(OUT_DIR / "X_concat.npy")
X_audio = X_concat[:, :768]
X_text  = X_concat[:, 768:]
y = np.load(OUT_DIR / "y.npy").astype(np.float32)
meta = pd.read_csv(OUT_DIR / "meta.csv")

groups = meta["speaker_id"].astype(str).values
languages = meta["language"].values


class FusionDataset(Dataset):
    def __init__(self, Xa, Xt, y):
        self.Xa = torch.from_numpy(Xa).float()
        self.Xt = torch.from_numpy(Xt).float()
        self.y  = torch.from_numpy(y).float()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.Xa[i], self.Xt[i], self.y[i]

class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=768, hidden=256, heads=4, dropout=0.25):
        super().__init__()

        self.audio_proj = nn.Linear(dim, hidden)
        self.text_proj  = nn.Linear(dim, hidden)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden,
            num_heads=heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(hidden)

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, xa, xt):
        qa = self.audio_proj(xa).unsqueeze(1)
        kv = self.text_proj(xt).unsqueeze(1)

        attn_out, _ = self.cross_attn(qa, kv, kv)
        h = self.norm(attn_out.squeeze(1))
        return self.head(h).squeeze(-1)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def train_one_fold(model, tr_loader, va_loader):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.SmoothL1Loss(beta=1.0)

    best_rmse, bad = float("inf"), 0
    best_state = None

    for _ in range(EPOCHS):
        model.train()
        for xa, xt, yb in tr_loader:
            xa, xt, yb = xa.to(DEVICE), xt.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xa, xt), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        preds, ys = [], []
        with torch.no_grad():
            for xa, xt, yb in va_loader:
                xa, xt = xa.to(DEVICE), xt.to(DEVICE)
                preds.append(model(xa, xt).cpu().numpy())
                ys.append(yb.numpy())

        preds = np.concatenate(preds)
        ys = np.concatenate(ys)
        val_rmse = rmse(ys, preds)

        if val_rmse < best_rmse - 1e-4:
            best_rmse = val_rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return best_rmse

gkf = GroupKFold(n_splits=N_SPLITS)
oof_pred = np.zeros_like(y)
fold_rmses = []

for fold, (tr, va) in enumerate(gkf.split(X_audio, y, groups=groups), 1):
    print(f"\n=== Fold {fold}/{N_SPLITS} ===")

    tr_ds = FusionDataset(X_audio[tr], X_text[tr], y[tr])
    va_ds = FusionDataset(X_audio[va], X_text[va], y[va])

    tr_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH, shuffle=False)

    model = CrossAttentionFusion(
        dim=768,
        hidden=HIDDEN,
        heads=N_HEADS,
        dropout=DROPOUT
    ).to(DEVICE)

    best_rmse = train_one_fold(model, tr_loader, va_loader)
    fold_rmses.append(best_rmse)

    model.eval()
    with torch.no_grad():
        preds = []
        for xa, xt, _ in va_loader:
            preds.append(model(xa.to(DEVICE), xt.to(DEVICE)).cpu().numpy())
    oof_pred[va] = np.concatenate(preds)

    print(f"Fold RMSE: {best_rmse:.4f}")

    del model
    torch.cuda.empty_cache()
    gc.collect()

print("\n=== CROSS-ATTENTION RESULTS ===")
print("Mean RMSE:", np.mean(fold_rmses))
print("Std  RMSE:", np.std(fold_rmses))
print("OOF RMSE :", rmse(y, oof_pred))
print("OOF MAE  :", mae(y, oof_pred))

SAVE_DIR = ROOT_DATA_PATH / "cross-attn-k5-huber"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "speaker_id": groups,
    "language": languages,
    "label": y,
    "pred_cross_attn": oof_pred
}).to_csv(SAVE_DIR / "cross_attn_k5_predictions.csv", index=False)

plt.figure(figsize=(7,7))
plt.scatter(y, np.clip(oof_pred, 0, 30), alpha=0.7)
plt.plot([0,30],[0,30],'r--')
plt.xlabel("Actual MMSE")
plt.ylabel("Predicted MMSE")
plt.title(f"Cross-Attention Fusion | RMSE={rmse(y,oof_pred):.2f}")
plt.grid(True)
plt.show()


Late Fusion - mlp on predictions

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from pathlib import Path
import gc

SEED = 42
N_SPLITS = 5

BATCH = 64 if torch.cuda.is_available() else 32
EPOCHS = 40
PATIENCE = 7

LR = 1e-3
WEIGHT_DECAY = 1e-3
HIDDEN = 16
DROPOUT = 0.2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

AST_PRED_PATH  = ROOT_DATA_PATH / "ast-k5-huber" / "ast_k5_predictions.csv"
TEXT_PRED_PATH = ROOT_DATA_PATH / "modernbert-k5-huber" / "modernbert_k20_predictions.csv"

df_a = pd.read_csv(AST_PRED_PATH)
df_t = pd.read_csv(TEXT_PRED_PATH)

# sanity
assert df_a["speaker_id"].nunique() == len(df_a)
assert df_t["speaker_id"].nunique() == len(df_t)

# merge
df = df_a[["speaker_id", "label", "pred_audio"]].merge(
    df_t[["speaker_id", "pred_text"]],
    on="speaker_id",
    how="inner"
)

print("Late-fusion rows:", len(df))

X = df[["pred_audio", "pred_text"]].values.astype(np.float32)
y = df["label"].values.astype(np.float32)
groups = df["speaker_id"].astype(str).values

class PredDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

class LateFusionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, HIDDEN),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def train_one_fold(model, tr_loader, va_loader):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.SmoothL1Loss(beta=1.0)

    best_rmse, bad = float("inf"), 0
    best_state = None

    for _ in range(EPOCHS):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        preds, ys = [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                xb = xb.to(DEVICE)
                preds.append(model(xb).cpu().numpy())
                ys.append(yb.numpy())

        preds = np.concatenate(preds)
        ys = np.concatenate(ys)
        val_rmse = rmse(ys, preds)

        if val_rmse < best_rmse - 1e-4:
            best_rmse = val_rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return best_rmse

gkf = GroupKFold(n_splits=N_SPLITS)
oof_pred = np.zeros_like(y)
fold_rmses = []

for fold, (tr, va) in enumerate(gkf.split(X, y, groups=groups), 1):
    print(f"\n=== Fold {fold}/{N_SPLITS} ===")

    tr_ds = PredDataset(X[tr], y[tr])
    va_ds = PredDataset(X[va], y[va])

    tr_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH, shuffle=False)

    model = LateFusionMLP().to(DEVICE)
    best_rmse = train_one_fold(model, tr_loader, va_loader)
    fold_rmses.append(best_rmse)

    model.eval()
    with torch.no_grad():
        preds = []
        for xb, _ in va_loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
    oof_pred[va] = np.concatenate(preds)

    print(f"Fold RMSE: {best_rmse:.4f}")

    del model
    torch.cuda.empty_cache()
    gc.collect()

print("\n=== LATE FUSION (MLP) RESULTS ===")
print("Mean RMSE:", np.mean(fold_rmses))
print("Std  RMSE:", np.std(fold_rmses))
print("OOF RMSE :", rmse(y, oof_pred))
print("OOF MAE  :", mae(y, oof_pred))

SAVE_DIR = ROOT_DATA_PATH / "late-fusion-mlp-k5-huber"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "speaker_id": groups,
    "label": y,
    "pred_late_mlp": oof_pred
}).to_csv(SAVE_DIR / "late_mlp_k5_predictions.csv", index=False)

plt.figure(figsize=(7,7))
plt.scatter(y, np.clip(oof_pred, 0, 30), alpha=0.7)
plt.plot([0,30],[0,30],'r--')
plt.xlabel("Actual MMSE")
plt.ylabel("Predicted MMSE")
plt.title(f"Late Fusion (MLP) | RMSE={rmse(y,oof_pred):.2f}")
plt.grid(True)
plt.show()
